# Cold-start short-video recommendation: end to end (Kaggle)

Settings (right-hand panel): **Accelerator: GPU T4 x2** (or P100) and **Internet: on**. Then **Save Version > Save & Run All (Commit)**, which runs in the background for up to 12 hours.

The run stops starting new steps after `MAX_HOURS` and saves what it has. To continue, add the previous version's output as an input (**Add Input > Your Work > this notebook**) and run again: finished runs are skipped.

Keep this notebook private. Its output holds features computed from MicroLens, and the dataset's licence does not allow passing on modified copies.

In [ ]:
REPO_URL = "https://github.com/FireCloud24/shortvideo-coldstart.git"
SEEDS = "0 1 2"
EXTRAS = 0  # 1 also runs English CLIP titles, LoRA and Qwen2-VL (slow; not needed for the finding)
IMAGE_LORA_STEPS = 3000
MAX_HOURS = 10  # start no new step after this; leaves time to save before the 12-hour limit

import glob, os
WORK = '/kaggle/temp/shortvideo-coldstart'
PERSIST_DIR = '/kaggle/working/persist'
os.makedirs('/kaggle/temp', exist_ok=True)
if not os.path.exists(WORK):
    !git clone -q $REPO_URL $WORK
%cd $WORK
!git log -1 --format='code version: %h %s'
!pip install -q -r requirements.txt
!pip uninstall -y -q torchao
import torch; print(torch.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')
assert torch.cuda.is_available(), 'No GPU: set Accelerator to GPU in the notebook settings'

## 1. Tests

In [ ]:
!python -m pytest -q

## 2. Download MicroLens-100K
About 710 MB, into `/kaggle/temp` (not kept in the output).

In [ ]:
!python scripts/download.py
!cat data/raw/manifest.json | head -20

## 3. Smoke test

In [ ]:
!python scripts/embed.py --model cnclip --limit 64
!python scripts/embed.py --model enclip --limit 64
!python scripts/train.py --config configs/sasrec_id.yaml --seeds 0 --set train.epochs=1 out_dir=results_smoke

## 4. Every experiment
Resumes from an earlier version's output when one is attached as an input.

In [ ]:
# The repository's results/ folder holds an earlier run; remove it so no old number survives.
!rm -rf results/runs results/finetune results/features results/lr_search results/*.md results/*.png
earlier = sorted(glob.glob('/kaggle/input/*/persist') + glob.glob('/kaggle/input/*/*/persist'))
if earlier and not os.path.isdir(f'{PERSIST_DIR}/results'):
    print('resuming from', earlier[-1])
    !mkdir -p $PERSIST_DIR
    !cp -r "{earlier[-1]}/." $PERSIST_DIR/
if os.path.isdir(f'{PERSIST_DIR}/results'):
    !mkdir -p results data/features checkpoints
    !cp -r $PERSIST_DIR/results/. results/
    !cp -r $PERSIST_DIR/features/. data/features/
    !cp -r $PERSIST_DIR/checkpoints/. checkpoints/
    !cp $PERSIST_DIR/run_all.log run_all.log
    !echo "=== resumed in a new session ===" >> run_all.log
!SEEDS="{SEEDS}" EXTRAS={EXTRAS} IMAGE_LORA_STEPS={IMAGE_LORA_STEPS} MAX_HOURS={MAX_HOURS} PERSIST_DIR="{PERSIST_DIR}" bash scripts/run_all.sh >> run_all.log 2>&1
!cp run_all.log results/run_all.log
!grep -E '^=== |seed [0-9]+: |^!!! |^wrote |cost: ' run_all.log

## 5. Report

In [ ]:
from IPython.display import Markdown, Image, display
for name in ('tradeoff', 'exposure', 'categories', 'warm_vs_cold', 'error_slices'):
    if os.path.exists(f'results/{name}.png'):
        display(Image(f'results/{name}.png'))
for name in ('lr_search/chosen', 'tradeoff', 'exposure', 'categories', 'results'):
    if os.path.exists(f'results/{name}.md'):
        display(Markdown(open(f'results/{name}.md').read()))

## 5b. Vector-length test
Retrains two ID + content models with their weights saved and multiplies the new videos' vectors by a factor, with nothing retrained. About half an hour.

In [ ]:
!bash scripts/run_scale_test.sh > scale_test.log 2>&1
!grep -E '^=== |as trained|^!!! |^wrote ' scale_test.log
from IPython.display import Markdown, Image, display
display(Image('results/scale_test/scale_test.png'))
display(Markdown(open('results/scale_test/scale_test.md').read()))

## 6. Save the results
`results.zip` holds only metrics, per-example ranks, tables, charts and the run log (no dataset files), so it can be committed. It appears under this version's **Output**.

In [ ]:
!cp -r results/. $PERSIST_DIR/results/
!cd $WORK && zip -q -r /kaggle/working/results.zip results
!ls -la /kaggle/working